In [7]:
import pandas as pd

# 아까 그레인저 2144개 파일 불러오기
df_granger = pd.read_csv('submission_GRANGER.csv')
df_adv = pd.read_csv('submission_SPEARMAN_04_9.csv')
# 교집합 구하기 (inner join)
# on=['leading_item_id', 'following_item_id'] 기준
df_intersection = pd.merge(
    df_adv[['leading_item_id', 'following_item_id']], 
    df_granger[['leading_item_id', 'following_item_id']], 
    on=['leading_item_id', 'following_item_id'], 
    how='inner'
)

# 값 채우기 (더미값)
df_intersection['value'] = 9999999999

df_intersection.to_csv('submission_INTERSECTION.csv', index=False)

print(f"🔥 [교집합] {len(df_intersection)}개 발견")

🔥 [교집합] 409개 발견


In [ ]:
import pandas as pd

# 아까 그레인저 2144개 파일 불러오기
df_granger = pd.read_csv('submission_GRANGER_005_9.csv')
df_adv = pd.read_csv('submission_SPEARMAN_04_9.csv')

# --- 변경된 부분: 합집합 구하기 (outer join) ---
# on=['leading_item_id', 'following_item_id'] 기준
df_union = pd.merge(
    df_adv[['leading_item_id', 'following_item_id']], 
    df_granger[['leading_item_id', 'following_item_id']], 
    on=['leading_item_id', 'following_item_id'], 
    how='outer'  # 교집합 대신 합집합을 위해 'outer' 사용
)

# 값 채우기 (더미값)
# 합집합 결과 DataFrame의 이름을 df_union으로 변경
df_union['value'] = 9999999999

df_union.to_csv('submission_UNION.csv', index=False) # 파일명도 UNION으로 변경

print(f"✨ [합집합] {len(df_union)}개 발견")

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from tqdm import tqdm

# 1. 데이터 및 1등 리스트(그레인저 2144개) 로드
df = pd.read_csv('train.csv')
target_list = pd.read_csv('submission_GRANGER.csv') # 점수 제일 좋았던 파일

print(f"🏆 챔피언 리스트 로드 완료: {len(target_list)}개 쌍")

# 전처리
df_grouped = df.groupby(['item_id', 'year', 'month'])['value'].sum().reset_index()
df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))
pivot_df = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)

# 2. XGBoost로 개별 예측 수행
predictions = []

for idx, row in tqdm(target_list.iterrows(), total=len(target_list), desc="XGBoosting"):
    lead_item = row['leading_item_id']
    follow_item = row['following_item_id']
    
    y = pivot_df[follow_item]
    x = pivot_df[lead_item]
    
    # 학습 데이터 생성 (Lag 1~6개월 데이터를 Feature로 사용)
    features = []
    targets = []
    
    # 데이터가 충분한 구간부터 학습
    for i in range(6, len(x)):
        obs = x.iloc[i-6:i].values[::-1] # 최근 6개월치 역순
        features.append(obs)
        targets.append(y.iloc[i])
    
    X_data = np.array(features)
    y_data = np.array(targets)
    
    # XGBoost 모델링
    model = xgb.XGBRegressor(
        n_estimators=200,    # 나무 개수 좀 늘림
        learning_rate=0.05,  # 학습률 낮춰서 꼼꼼하게
        max_depth=4, 
        n_jobs=1,
        objective='reg:squarederror'
    )
    model.fit(X_data, y_data)
    
    # 2025년 8월 예측 (7월 기준 과거 6개월 데이터 입력)
    last_input = x.iloc[-6:].values[::-1].reshape(1, -1)
    pred_val = model.predict(last_input)[0]
    predictions.append(max(0, int(pred_val)))

# 예측값 컬럼 추가
target_list['value'] = predictions

# ---------------------------------------------------------
# 🔥 [핵심 전략] Median Ensemble (중앙값 통일)
# 같은 후행 품목(B)을 바라보는 여러 예측값들의 중앙값을 취함
# 이상한 예측값 하나가 점수 망치는 걸 방지함
# ---------------------------------------------------------
print("\n⚙️ 앙상블(Median) 적용 중...")

# transform을 쓰면 그룹별 중앙값을 각 행에 쫙 뿌려줌
target_list['ensemble_value'] = target_list.groupby('following_item_id')['value'].transform('median')

# 정수 변환 후 덮어쓰기
target_list['value'] = target_list['ensemble_value'].round().astype(int)

# 필요 없는 컬럼 버리기
final_submission = target_list[['leading_item_id', 'following_item_id', 'value']]

# 저장
final_submission.to_csv('submission_FINAL_WINNER.csv', index=False)

print(f"\n✅ 최종 파일 생성 완료: submission_FINAL_WINNER.csv")
print("이 파일은 [F1 점수(0.23) 유지] + [NMAE 점수 획득] 전략입니다.")

🏆 챔피언 리스트 로드 완료: 912개 쌍


XGBoosting: 100%|██████████| 912/912 [00:07<00:00, 115.04it/s]


⚙️ 앙상블(Median) 적용 중...

✅ 최종 파일 생성 완료: submission_FINAL_WINNER.csv
이 파일은 [F1 점수(0.23) 유지] + [NMAE 점수 획득] 전략입니다.


In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from statsmodels.tsa.stattools import grangercausalitytests
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# 1. 데이터 로드
print("📂 데이터 및 리스트 로드 중...")
train_df = pd.read_csv('train.csv')
granger_df = pd.read_csv('submission_GRANGER.csv') # 2144개 쌍 리스트

# 전처리
df_grouped = train_df.groupby(['item_id', 'year', 'month'])['value'].sum().reset_index()
df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))
pivot_df = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)

# ---------------------------------------------------------
# 2. 원본 데이터 기반 F-statistic 계산 (진짜 대장 찾기)
# ---------------------------------------------------------
print("📊 원본 데이터로 인과관계 강도(F-score) 계산 중...")
f_scores = []
MAX_LAG = 6

for idx, row in tqdm(granger_df.iterrows(), total=len(granger_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    y = pivot_df[follow]
    x = pivot_df[lead]
    
    # 데이터 합치기
    data = pd.concat([y, x], axis=1)
    
    best_f = 0
    if data.std().min() > 0: # 데이터가 살아있으면
        try:
            # Granger 테스트
            gc_res = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
            # Lag 1~6 중 최대 F-stat 추출
            for lag in range(1, MAX_LAG + 1):
                f_stat = gc_res[lag][0]['ssr_ftest'][0]
                if f_stat > best_f:
                    best_f = f_stat
        except:
            pass
    
    f_scores.append(best_f)

granger_df['f_score'] = f_scores

# ---------------------------------------------------------
# 3. 각 타겟별 Top-3 선정 (Elite Pairs)
# ---------------------------------------------------------
TOP_N = 3
print(f"\n🏆 타겟별 F-score 상위 {TOP_N}개 선정 중...")

# 정렬 후 Top-3만 남긴 새로운 데이터프레임 생성
elite_df = granger_df.sort_values(by=['following_item_id', 'f_score'], ascending=[True, False])
elite_df = elite_df.groupby('following_item_id').head(TOP_N).copy()

print(f"   -> 총 {len(granger_df)}개 중 {len(elite_df)}개 쌍을 예측 모델로 사용")

# ---------------------------------------------------------
# 4. Top-3에 대해서만 XGBoost 학습 및 예측
# ---------------------------------------------------------
print("\n🤖 Elite 쌍에 대해 XGBoost 예측 수행...")

predictions = []

for idx, row in tqdm(elite_df.iterrows(), total=len(elite_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    y = pivot_df[follow]
    x = pivot_df[lead]
    
    # --- XGBoost 데이터셋 생성 (Lag 6) ---
    features = []
    targets = []
    
    for i in range(6, len(x)):
        obs = x.iloc[i-6:i].values[::-1]
        features.append(obs)
        targets.append(y.iloc[i])
        
    X_data = np.array(features)
    y_data = np.array(targets)
    
    # 모델 학습
    model = xgb.XGBRegressor(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        n_jobs=1,
        objective='reg:squarederror'
    )
    model.fit(X_data, y_data)
    
    # 예측 (2025년 8월)
    last_input = x.iloc[-6:].values[::-1].reshape(1, -1)
    pred_val = model.predict(last_input)[0]
    
    predictions.append(max(0, int(pred_val)))

elite_df['predicted_value'] = predictions

# ---------------------------------------------------------
# 5. 앙상블 (Top-3 평균) 및 전체 전파 (Broadcasting)
# ---------------------------------------------------------
print("\n🔗 Top-3 평균값을 전체 리스트에 공유 중...")

# 1) Elite들의 타겟별 평균값 계산
target_values = elite_df.groupby('following_item_id')['predicted_value'].mean()

# 2) 전체 리스트(granger_df)에 매핑
# map을 쓰면 2144개 전체 행에 대해, 자기 타겟(B)에 해당하는 Top-3 평균값이 들어감
granger_df['final_value'] = granger_df['following_item_id'].map(target_values)

# 3) 결측치 처리 (혹시 Top-3가 하나도 없는 경우 대비 - 거의 없겠지만)
granger_df['final_value'] = granger_df['final_value'].fillna(0)

# 4) 정수 변환 및 정리
granger_df['value'] = granger_df['final_value'].round().astype(int)
final_submission = granger_df[['leading_item_id', 'following_item_id', 'value']]

# 저장
filename = 'submission_REAL_FINAL_TOP3_XGB.csv'
final_submission.to_csv(filename, index=False)

print(f"\n✅ 최종 완료: {filename}")
print("   - 구조: Granger 2144개 (F1 점수 보존)")
print("   - 값: Top-3 F-score Elite의 XGB 예측 평균 (NMAE 점수 최적화)")

📂 데이터 및 리스트 로드 중...
📊 원본 데이터로 인과관계 강도(F-score) 계산 중...


100%|██████████| 2144/2144 [00:04<00:00, 448.08it/s]



🏆 타겟별 F-score 상위 3개 선정 중...
   -> 총 2144개 중 297개 쌍을 예측 모델로 사용

🤖 Elite 쌍에 대해 XGBoost 예측 수행...


100%|██████████| 297/297 [00:01<00:00, 199.32it/s]


🔗 Top-3 평균값을 전체 리스트에 공유 중...

✅ 최종 완료: submission_REAL_FINAL_TOP3_XGB.csv
   - 구조: Granger 2144개 (F1 점수 보존)
   - 값: Top-3 F-score Elite의 XGB 예측 평균 (NMAE 점수 최적화)


In [29]:
import pandas as pd
import numpy as np
import xgboost as xgb
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# 1. 데이터 로드
print("📂 데이터 준비 중...")
train_df = pd.read_csv('train.csv')
train_df = train_df[train_df['value'] != 0]
granger_df = pd.read_csv('submission_GRANGER.csv') # 2144개 후보 리스트

# 전처리
df_grouped = train_df.groupby(['item_id', 'year', 'month'])['value'].sum().reset_index()
df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))
pivot_df = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)

# [검증용 데이터 분리]
# Train: ~ 2025년 6월 (마지막 한 달 제외)
train_pivot = pivot_df.iloc[:-1] 

# Valid: 2025년 7월 (정답지)
val_actual = pivot_df.iloc[-1]

print(f"학습 기간: {train_pivot.index[0].date()} ~ {train_pivot.index[-1].date()}")
print(f"검증 타겟: {val_actual.name.date()} (이걸 맞춰야 함)")

# ---------------------------------------------------------
# 2. F-Score 계산 (Train 데이터로만 계산!)
# ---------------------------------------------------------
print("\n📊 (Train only) 인과관계 F-score 계산 중...")
f_scores = []
MAX_LAG = 6

for idx, row in tqdm(granger_df.iterrows(), total=len(granger_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    y = train_pivot[follow] # 6월까지만 봄
    x = train_pivot[lead]   # 6월까지만 봄
    
    data = pd.concat([y, x], axis=1)
    best_f = 0
    
    if data.std().min() > 0:
        try:
            gc_res = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
            for lag in range(1, MAX_LAG + 1):
                f_stat = gc_res[lag][0]['ssr_ftest'][0]
                if f_stat > best_f:
                    best_f = f_stat
        except:
            pass
    f_scores.append(best_f)

granger_df['f_score'] = f_scores

# ---------------------------------------------------------
# 3. Top-3 선정 & XGBoost 예측
# ---------------------------------------------------------
TOP_N = 3
print(f"\n🏆 Top-{TOP_N} 선정 및 예측 수행...")

# F-score 기준 정렬 및 상위 N개 추출
elite_df = granger_df.sort_values(by=['following_item_id', 'f_score'], ascending=[True, False])
elite_df = elite_df.groupby('following_item_id').head(TOP_N).copy()

predictions = []
actuals = [] # 정답 담을 리스트

print("\n🚀 Feature Engineering 강화된 XGBoost 학습 시작...")

for idx, row in tqdm(elite_df.iterrows(), total=len(elite_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    # [수정 포인트] 학습에는 pivot_df가 아니라 train_pivot(6월까지)을 써야 함!
    y = train_pivot[follow] # 타겟 (B)
    x = train_pivot[lead]   # 선행 (A)
    
    features = []
    targets = []
    
    # 데이터 구성 (Lag 6부터 시작)
    for i in range(6, len(x)):
        # 기준 시점
        current_date = x.index[i]
        
        # [Feature 1] Lag 1~6 (기존)
        lags = x.iloc[i-6:i].values[::-1]
        
        # [Feature 2] 시간 정보
        month = current_date.month
        year = current_date.year
        
        # [Feature 3] 이동 평균 & 변동성
        roll_mean_3 = np.mean(lags[:3])
        roll_mean_6 = np.mean(lags[:6])
        roll_std_6 = np.std(lags[:6])
        
        # Feature 합치기
        obs = np.concatenate([lags, [month, year, roll_mean_3, roll_mean_6, roll_std_6]])
        
        features.append(obs)
        targets.append(y.iloc[i])
        
    X_data = np.array(features)
    y_data = np.array(targets)
    
    # 모델 학습
    model = xgb.XGBRegressor(
        n_estimators=200, 
        learning_rate=0.05, 
        max_depth=5,
        n_jobs=1
    )
    model.fit(X_data, y_data)
    
    # -----------------------------------------------------
    # 2025년 7월 예측 (Input: 1~6월 데이터)
    # -----------------------------------------------------
    # 1. Lags (x는 train_pivot이므로 마지막 데이터는 6월임)
    last_lags = x.iloc[-6:].values[::-1]
    
    # 2. Time (예측 대상 시점은 2025년 7월)
    pred_month = 7
    pred_year = 2025
    
    # 3. Rolling Stats
    last_mean_3 = np.mean(last_lags[:3])
    last_mean_6 = np.mean(last_lags[:6])
    last_std_6 = np.std(last_lags[:6])
    
    # Input 합치기
    final_input = np.concatenate([last_lags, [pred_month, pred_year, last_mean_3, last_mean_6, last_std_6]])
    final_input = final_input.reshape(1, -1)
    
    # 예측값 저장
    pred_val = max(0, int(model.predict(final_input)[0]))
    predictions.append(pred_val)
    
    # [수정 포인트] 실제 정답(7월 값) 저장 (이게 빠져서 에러났었음)
    actuals.append(val_actual[follow])

# 결과 담기
elite_df['pred_jul'] = predictions
elite_df['actual_jul'] = actuals

# ---------------------------------------------------------
# 4. 앙상블 적용 및 성능 평가
# ---------------------------------------------------------
print("\n⚖️ 성능 평가 (NMAE) 계산 중...")

# 1) 개별 모델(XGB)만 썼을 때의 오차
errors_individual = []
for idx, row in elite_df.iterrows():
    true_val = row['actual_jul']
    pred_val = row['pred_jul']
    err = abs(true_val - pred_val) / (abs(true_val) + 1e-6)
    errors_individual.append(min(err, 1.0))

nmae_individual = np.mean(errors_individual)

# 2) 앙상블(Top-3 Mean) 적용 후 오차
# 타겟별로 평균 예측값 계산
ensemble_preds = elite_df.groupby('following_item_id')['pred_jul'].mean()

errors_ensemble = []
for target, pred_mean in ensemble_preds.items():
    true_val = val_actual[target]
    pred_val = int(round(pred_mean))
    
    err = abs(true_val - pred_val) / (abs(true_val) + 1e-6)
    errors_ensemble.append(min(err, 1.0))

nmae_ensemble = np.mean(errors_ensemble)

# ---------------------------------------------------------
# 5. 결과 리포트
# ---------------------------------------------------------
print("\n" + "="*40)
print(f"   [7월 예측 검증 결과]")
print("="*40)
print(f"👉 개별 XGBoost 평균 NMAE : {nmae_individual:.4f}")
print(f"👉 Top-{TOP_N} 앙상블 평균 NMAE  : {nmae_ensemble:.4f}")
print("-" * 40)

if nmae_ensemble < nmae_individual:
    print("✅ 결론: 앙상블(평균) 전략이 더 우수함! (오차 감소)")
    improvement = (nmae_individual - nmae_ensemble) / nmae_individual * 100
    print(f"   (약 {improvement:.1f}% 성능 향상)")
else:
    print("❌ 결론: 앙상블이 오히려 안 좋음. Top-1만 쓰는 게 나을 수도?")

# 점수 환산 (NMAE 점수만)
score_nmae = 1 - nmae_ensemble
print(f"\n🎯 예상 NMAE 점수(0.4 만점 기준): {score_nmae * 0.4:.4f}점 획득 예상")

📂 데이터 준비 중...
학습 기간: 2022-01-01 ~ 2025-06-01
검증 타겟: 2025-07-01 (이걸 맞춰야 함)

📊 (Train only) 인과관계 F-score 계산 중...


100%|██████████| 1807/1807 [00:04<00:00, 426.85it/s]



🏆 Top-3 선정 및 예측 수행...

🚀 Feature Engineering 강화된 XGBoost 학습 시작...


100%|██████████| 273/273 [00:03<00:00, 77.84it/s]


⚖️ 성능 평가 (NMAE) 계산 중...

   [7월 예측 검증 결과]
👉 개별 XGBoost 평균 NMAE : 0.4979
👉 Top-3 앙상블 평균 NMAE  : 0.4941
----------------------------------------
✅ 결론: 앙상블(평균) 전략이 더 우수함! (오차 감소)
   (약 0.8% 성능 향상)

🎯 예상 NMAE 점수(0.4 만점 기준): 0.2024점 획득 예상


In [36]:
import pandas as pd
import numpy as np
import xgboost as xgb
from statsmodels.tsa.stattools import grangercausalitytests
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# 1. 데이터 로드
print("📂 데이터 준비 중...")
train_df = pd.read_csv('train.csv')
train_df = train_df[train_df['value'] != 0]
granger_df = pd.read_csv('submission_GRANGER.csv') 

# ---------------------------------------------------------
# [변경 1] 전처리: Value(합계)와 Seq(최대값=거래횟수)를 동시에 집계
# ---------------------------------------------------------
df_grouped = train_df.groupby(['item_id', 'year', 'month']).agg({
    'value': 'sum',
    'seq': 'max'  # 해당 월의 마지막 일련번호 (= 거래 횟수)
}).reset_index()

df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))

# Pivot 테이블 2개 생성
pivot_val = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)
pivot_seq = df_grouped.pivot(index='date', columns='item_id', values='seq').fillna(0)

# [검증용 데이터 분리]
# Train: ~ 6월
train_val = pivot_val.iloc[:-1] 
train_seq = pivot_seq.iloc[:-1] # Seq 데이터도 6월까지만

# Valid: 7월 (정답지)
val_actual = pivot_val.iloc[-1]

print(f"학습 기간: {train_val.index[0].date()} ~ {train_val.index[-1].date()}")

# ---------------------------------------------------------
# 2. F-Score 계산 (Value 기준)
# ---------------------------------------------------------
# Seq가 추가돼도 인과관계 찾기(짝꿍 찾기)는 Value로 하는 게 정확함
print("\n📊 (Train only) 인과관계 F-score 계산 중...")
f_scores = []
MAX_LAG = 6

for idx, row in tqdm(granger_df.iterrows(), total=len(granger_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    y = train_val[follow]
    x = train_val[lead]
    
    data = pd.concat([y, x], axis=1)
    best_f = 0
    if data.std().min() > 0:
        try:
            gc_res = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
            for lag in range(1, MAX_LAG + 1):
                f_stat = gc_res[lag][0]['ssr_ftest'][0]
                if f_stat > best_f: best_f = f_stat
        except: pass
    f_scores.append(best_f)

granger_df['f_score'] = f_scores

# Top-3 선정
TOP_N = 3
print(f"\n🏆 Top-{TOP_N} 선정 및 예측 수행...")
elite_df = granger_df.sort_values(by=['following_item_id', 'f_score'], ascending=[True, False])
elite_df = elite_df.groupby('following_item_id').head(TOP_N).copy()

predictions = []
actuals = []

print("\n🚀 Feature Engineering (Value + Seq) 학습 시작...")

for idx, row in tqdm(elite_df.iterrows(), total=len(elite_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    # Value 데이터
    y_val = train_val[follow] 
    x_val = train_val[lead]   
    
    # [필수] Seq 데이터 (선행 품목의 거래 빈도)
    x_seq = train_seq[lead] 
    
    features = []
    targets = []
    
    # 데이터 구성 (Lag 6부터 시작)
    for i in range(6, len(x_val)):
        current_date = x_val.index[i]
        
        # 1. Value Lags (6개)
        lags_val = x_val.iloc[i-6:i].values[::-1]
        
        # 2. Seq Lags (6개) -> [이게 빠지면 11개가 됨]
        lags_seq = x_seq.iloc[i-6:i].values[::-1]
        
        # 3. Time (2개)
        month = current_date.month
        year = current_date.year
        
        # 4. Rolling Stats (2개)
        roll_mean_3 = np.mean(lags_val[:3])
        roll_mean_6 = np.mean(lags_val[:6])
        # (Std6은 제외했습니다. Seq랑 Value를 같이 쓰면 16개가 딱 깔끔해서)
        
        # ---------------------------------------------------------
        # [중요] 여기서 합칠 때 lags_seq가 꼭 들어가야 함!
        # 6(Val) + 6(Seq) + 2(Time) + 2(Roll) = 16개
        # ---------------------------------------------------------
        obs = np.concatenate([
            lags_val,      # 6개
            lags_seq,      # 6개
            [month, year, roll_mean_3, roll_mean_6] # 4개
        ])
        
        features.append(obs)
        targets.append(y_val.iloc[i])
        
    X_data = np.array(features)
    y_data = np.array(targets)
    
    # 확인용 (첫 번째 줄만 길이 출력) - 16 나오는지 확인
    if idx == 0:
        print(f"Feature 개수 확인: {len(X_data[0])}개 (16개여야 정상)")

    # 모델 학습
    model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6, n_jobs=1
    )
    model.fit(X_data, y_data)
    
    # -----------------------------------------------------
    # 2025년 7월 예측 (Input도 16개 맞춰야 함)
    # -----------------------------------------------------
    last_lags_val = x_val.iloc[-6:].values[::-1]
    last_lags_seq = x_seq.iloc[-6:].values[::-1] # 여기도 필수
    
    pred_month = 7
    pred_year = 2025
    
    last_mean_3 = np.mean(last_lags_val[:3])
    last_mean_6 = np.mean(last_lags_val[:6])
    
    # Input 합치기 (순서 중요: Val -> Seq -> Time -> Roll)
    final_input = np.concatenate([
        last_lags_val, 
        last_lags_seq, 
        [pred_month, pred_year, last_mean_3, last_mean_6]
    ])
    final_input = final_input.reshape(1, -1)
    
    pred_val = max(0, int(model.predict(final_input)[0]))
    predictions.append(pred_val)
    actuals.append(val_actual[follow])

elite_df['pred_jul'] = predictions
elite_df['actual_jul'] = actuals

# ---------------------------------------------------------
# 평가
# ---------------------------------------------------------
print("\n⚖️ 성능 평가 (NMAE) 계산 중...")

# 앙상블 적용
ensemble_preds = elite_df.groupby('following_item_id')['pred_jul'].mean()
errors_ensemble = []

for target, pred_mean in ensemble_preds.items():
    true_val = val_actual[target]
    pred_val = int(round(pred_mean))
    err = abs(true_val - pred_val) / (abs(true_val) + 1e-6)
    errors_ensemble.append(min(err, 1.0))

nmae_ensemble = np.mean(errors_ensemble)

print("="*40)
print(f"👉 [Value + Seq + Time] Top-{TOP_N} 앙상블 NMAE : {nmae_ensemble:.4f}")
print("="*40)

📂 데이터 준비 중...
학습 기간: 2022-01-01 ~ 2025-06-01

📊 (Train only) 인과관계 F-score 계산 중...


100%|██████████| 1807/1807 [00:04<00:00, 436.61it/s]



🏆 Top-3 선정 및 예측 수행...

🚀 Feature Engineering (Value + Seq) 학습 시작...


100%|██████████| 273/273 [00:04<00:00, 66.79it/s]


⚖️ 성능 평가 (NMAE) 계산 중...
👉 [Value + Seq + Time] Top-3 앙상블 NMAE : 0.4871


In [38]:
import numpy as np

# =========================================================
# [사용자 입력] 네가 분석한 고정 수치
# =========================================================
P = 1807      # 제출 개수
TP = 680      # 맞춘 개수 (True Positive)
GT = 1371     # 전체 정답 개수 (Ground Truth)

# =========================================================
# [자동 계산] 파생 변수
# =========================================================
FP = P - TP          # 냈는데 틀린 것 (좀비 포함) -> 1127개
FN = GT - TP         # 안 냈는데 정답인 것 -> 691개
Union = P + FN       # 합집합 크기 (NMAE 분모) -> 2498개

print(f"📊 [시나리오 설정]")
print(f"   - 제출(P): {P} | 정답(G): {GT}")
print(f"   - TP: {TP} (37.6%) | FP: {FP} (62.4%) | FN: {FN}")
print("="*60)
print(f"{'순수 모델 오차(Error)':^20} | {'최종 NMAE':^15} | {'F1 점수':^15} | {'🏆 예상 총점':^15}")
print("-" * 60)

# =========================================================
# [시뮬레이션] 모델이 값을 얼마나 잘 맞추냐에 따른 점수 변화
# =========================================================
# model_error: TP에 대한 순수 오차율 (0.0 ~ 1.0)
# 0.9: 값을 거의 못 맞춤 (현재 상태 추정)
# 0.5: 반타작 (XGBoost 기본)
# 0.3: 고수 (로그변환 + 앙상블 성공 시)

for model_error in [0.9, 0.7, 0.53, 0.4, 0.2]:
    
    # 1. NMAE 계산
    # (TP오차합 + FP감점 + FN감점) / 전체합집합
    sum_tp_error = TP * model_error
    penalty_fp = FP * 1.0
    penalty_fn = FN * 1.0
    
    total_nmae = (sum_tp_error + penalty_fp + penalty_fn) / Union
    score_nmae = 1 - total_nmae # NMAE 점수 (1점 만점 기준)
    
    # 2. F1 계산
    precision = TP / P
    recall = TP / GT
    f1 = 2 * (precision * recall) / (precision + recall)
    
    # 3. 최종 총점 (0.6 * F1 + 0.4 * NMAE_Score)
    final_score = (0.6 * f1) + (0.4 * score_nmae)
    
    # 출력
    note = ""
    if model_error == 0.53: note = " (👈 현재 추정)"
    if model_error == 0.4: note = " (🎯 1차 목표)"
    
    print(f"{model_error:^20.2f} | {total_nmae:^15.4f} | {f1:^15.4f} | {final_score:^15.4f}{note}")

print("-" * 60)
print("💡 해석:")
print("1. 현재 구조(1807개 제출)에서 FP(1127개)가 너무 많아 기본 감점을 깔고 갑니다.")
print("2. 모델이 값을 기가 막히게 맞춰도(오차 0.2), 총점은 0.36점에 불과합니다.")
print("3. 결론: FP(1127개)를 줄이지 않으면(필터링) 점수 상승에 한계가 있습니다.")

📊 [시나리오 설정]
   - 제출(P): 1807 | 정답(G): 1371
   - TP: 680 (37.6%) | FP: 1127 (62.4%) | FN: 691
  순수 모델 오차(Error)    |     최종 NMAE     |      F1 점수      |     🏆 예상 총점    
------------------------------------------------------------
        0.90         |     0.9728      |     0.4279      |     0.2677     
        0.70         |     0.9183      |     0.4279      |     0.2894     
        0.53         |     0.8721      |     0.4279      |     0.3079      (👈 현재 추정)
        0.40         |     0.8367      |     0.4279      |     0.3221      (🎯 1차 목표)
        0.20         |     0.7822      |     0.4279      |     0.3439     
------------------------------------------------------------
💡 해석:
1. 현재 구조(1807개 제출)에서 FP(1127개)가 너무 많아 기본 감점을 깔고 갑니다.
2. 모델이 값을 기가 막히게 맞춰도(오차 0.2), 총점은 0.36점에 불과합니다.
3. 결론: FP(1127개)를 줄이지 않으면(필터링) 점수 상승에 한계가 있습니다.


In [30]:
import pandas as pd
import numpy as np
import xgboost as xgb
from statsmodels.tsa.stattools import grangercausalitytests
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# 1. 데이터 로드 (전체 데이터 사용)
print("📂 데이터 준비 중 (전체 데이터)...")
train_df = pd.read_csv('train.csv')
train_df = train_df[train_df['value'] != 0] # 사용자가 추가한 필터링 유지
granger_df = pd.read_csv('submission_GRANGER.csv') 

# ---------------------------------------------------------
# 전처리: Value(합계)와 Seq(최대값) 집계
# ---------------------------------------------------------
df_grouped = train_df.groupby(['item_id', 'year', 'month']).agg({
    'value': 'sum',
    'seq': 'max'
}).reset_index()

df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))

# Pivot 생성 (Split 없이 전체 데이터 사용)
pivot_val = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)
pivot_seq = df_grouped.pivot(index='date', columns='item_id', values='seq').fillna(0)

print(f"학습 기간: {pivot_val.index[0].date()} ~ {pivot_val.index[-1].date()} (7월 포함)")

# ---------------------------------------------------------
# 2. F-Score 계산 (전체 데이터 기준 재산정)
# ---------------------------------------------------------
print("\n📊 (Full Data) 인과관계 F-score 계산 중...")
f_scores = []
MAX_LAG = 6

for idx, row in tqdm(granger_df.iterrows(), total=len(granger_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    y = pivot_val[follow]
    x = pivot_val[lead]
    
    data = pd.concat([y, x], axis=1)
    best_f = 0
    if data.std().min() > 0:
        try:
            gc_res = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
            for lag in range(1, MAX_LAG + 1):
                f_stat = gc_res[lag][0]['ssr_ftest'][0]
                if f_stat > best_f: best_f = f_stat
        except: pass
    f_scores.append(best_f)

granger_df['f_score'] = f_scores

# Top-3 선정
TOP_N = 3
print(f"\n🏆 Top-{TOP_N} 선정 및 예측 수행...")
elite_df = granger_df.sort_values(by=['following_item_id', 'f_score'], ascending=[True, False])
elite_df = elite_df.groupby('following_item_id').head(TOP_N).copy()

predictions = []

print("\n🚀 Feature Engineering (Value + Seq) 학습 및 8월 예측...")

for idx, row in tqdm(elite_df.iterrows(), total=len(elite_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    # 전체 데이터 사용
    y_val = pivot_val[follow] 
    x_val = pivot_val[lead]   
    x_seq = pivot_seq[lead] 
    
    features = []
    targets = []
    
    # 데이터 구성 (Lag 6부터 끝까지)
    for i in range(6, len(x_val)):
        current_date = x_val.index[i]
        
        lags_val = x_val.iloc[i-6:i].values[::-1]
        lags_seq = x_seq.iloc[i-6:i].values[::-1]
        
        month = current_date.month
        year = current_date.year
        
        roll_mean_3 = np.mean(lags_val[:3])
        roll_mean_6 = np.mean(lags_val[:6])
        
        obs = np.concatenate([
            lags_val,      # 6개
            lags_seq,      # 6개
            [month, year, roll_mean_3, roll_mean_6] # 4개
        ])
        
        features.append(obs)
        targets.append(y_val.iloc[i])
        
    X_data = np.array(features)
    y_data = np.array(targets)
    
    # 모델 학습
    model = xgb.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=6, n_jobs=1
    )
    model.fit(X_data, y_data)
    
    # -----------------------------------------------------
    # 2025년 8월 예측 (Input: 2025년 2월 ~ 7월 데이터)
    # -----------------------------------------------------
    last_lags_val = x_val.iloc[-6:].values[::-1]
    last_lags_seq = x_seq.iloc[-6:].values[::-1]
    
    # [변경] 예측 시점: 8월
    pred_month = 8
    pred_year = 2025
    
    last_mean_3 = np.mean(last_lags_val[:3])
    last_mean_6 = np.mean(last_lags_val[:6])
    
    final_input = np.concatenate([
        last_lags_val, 
        last_lags_seq, 
        [pred_month, pred_year, last_mean_3, last_mean_6]
    ])
    final_input = final_input.reshape(1, -1)
    
    pred_val = max(0, int(model.predict(final_input)[0]))
    predictions.append(pred_val)

elite_df['pred_aug'] = predictions

# ---------------------------------------------------------
# 4. 앙상블 (Top-3 Mean) 및 전체 전파 (Broadcasting)
# ---------------------------------------------------------
print("\n🔗 Top-3 평균값을 전체 리스트에 공유 중...")

# 1) Elite들의 타겟별 8월 평균값 계산
target_values = elite_df.groupby('following_item_id')['pred_aug'].mean()

# 2) 전체 리스트(granger_df)에 매핑
# 2144개 전체 행에 대해, 해당 타겟의 Top-3 평균값을 넣음
granger_df['final_value'] = granger_df['following_item_id'].map(target_values)

# 3) 결측치 처리 (혹시 Top-3가 없는 경우 0 처리)
granger_df['final_value'] = granger_df['final_value'].fillna(0)

# 4) 정수 변환 및 컬럼 정리
granger_df['value'] = granger_df['final_value'].round().astype(int)
final_submission = granger_df[['leading_item_id', 'following_item_id', 'value']]

# 저장
filename = 'submission_FINAL_AUG_SEQ.csv'
final_submission.to_csv(filename, index=False)

print(f"\n✅ 최종 제출 파일 생성 완료: {filename}")
print(f"   - 총 {len(final_submission)}개 행")
print("   - Feature: Value(6) + Seq(6) + Time(2) + Rolling(2) = 16개")

📂 데이터 준비 중 (전체 데이터)...
학습 기간: 2022-01-01 ~ 2025-07-01 (7월 포함)

📊 (Full Data) 인과관계 F-score 계산 중...


100%|██████████| 1807/1807 [00:04<00:00, 438.87it/s]



🏆 Top-3 선정 및 예측 수행...

🚀 Feature Engineering (Value + Seq) 학습 및 8월 예측...


100%|██████████| 273/273 [00:04<00:00, 65.74it/s]


🔗 Top-3 평균값을 전체 리스트에 공유 중...

✅ 최종 제출 파일 생성 완료: submission_FINAL_AUG_SEQ.csv
   - 총 1807개 행
   - Feature: Value(6) + Seq(6) + Time(2) + Rolling(2) = 16개


In [31]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.stattools import grangercausalitytests
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# 1. 데이터 로드
print("📂 데이터 준비 중...")
train_df = pd.read_csv('train.csv')
train_df = train_df[train_df['value'] != 0]
granger_df = pd.read_csv('submission_GRANGER.csv') 

# 전처리
df_grouped = train_df.groupby(['item_id', 'year', 'month']).agg({
    'value': 'sum',
    'seq': 'max'
}).reset_index()

df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))

pivot_val = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)
pivot_seq = df_grouped.pivot(index='date', columns='item_id', values='seq').fillna(0)

# [핵심 1] 값에 로그 씌우기 (Log1p: log(x+1) -> 0인 값도 에러 안 나게)
# 학습할 때는 로그값으로 하고, 나중에 expm1로 복구함
pivot_val_log = np.log1p(pivot_val)

# ---------------------------------------------------------
# 2. F-Score 계산 (로그 변환된 값으로 계산해도 됨, 혹은 원본 유지)
# ---------------------------------------------------------
# F-score는 순위 매기기용이라 원본 값 써도 무방하지만, 일관성을 위해 원본 사용 권장
print("\n📊 인과관계 F-score 계산 중...")
f_scores = []
MAX_LAG = 6

for idx, row in tqdm(granger_df.iterrows(), total=len(granger_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    y = pivot_val[follow] # 그레인저는 원본 스케일로 확인
    x = pivot_val[lead]
    
    data = pd.concat([y, x], axis=1)
    best_f = 0
    if data.std().min() > 0:
        try:
            gc_res = grangercausalitytests(data, maxlag=MAX_LAG, verbose=False)
            for lag in range(1, MAX_LAG + 1):
                f_stat = gc_res[lag][0]['ssr_ftest'][0]
                if f_stat > best_f: best_f = f_stat
        except: pass
    f_scores.append(best_f)

granger_df['f_score'] = f_scores

# Top-3 선정
TOP_N = 3
elite_df = granger_df.sort_values(by=['following_item_id', 'f_score'], ascending=[True, False])
elite_df = elite_df.groupby('following_item_id').head(TOP_N).copy()

predictions = []

print("\n🚀 [Log 변환] XGBoost(50%) + Linear(50%) 앙상블 학습...")

for idx, row in tqdm(elite_df.iterrows(), total=len(elite_df)):
    lead = row['leading_item_id']
    follow = row['following_item_id']
    
    # [핵심 2] 학습 데이터는 로그 변환된 값 사용!
    y_log = pivot_val_log[follow] 
    x_log = pivot_val_log[lead]   
    x_seq = pivot_seq[lead] # Seq는 로그 안 해도 됨 (값이 작아서)
    
    features = []
    targets = []
    
    # 데이터 구성
    for i in range(6, len(x_log)):
        current_date = x_log.index[i]
        
        lags_val = x_log.iloc[i-6:i].values[::-1] # 로그된 Lags
        lags_seq = x_seq.iloc[i-6:i].values[::-1]
        
        month = current_date.month
        year = current_date.year
        
        roll_mean_3 = np.mean(lags_val[:3])
        roll_mean_6 = np.mean(lags_val[:6])
        
        obs = np.concatenate([
            lags_val, 
            lags_seq, 
            [month, year, roll_mean_3, roll_mean_6]
        ])
        
        features.append(obs)
        targets.append(y_log.iloc[i]) # 타겟도 로그값
        
    X_data = np.array(features)
    y_data = np.array(targets)
    
    # --- 모델 1: XGBoost ---
    model_xgb = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, n_jobs=1)
    model_xgb.fit(X_data, y_data)
    
    # --- 모델 2: Linear Regression (추세 반영용) ---
    model_lr = LinearRegression()
    model_lr.fit(X_data, y_data)
    
    # --- 8월 예측 ---
    last_lags_val = x_log.iloc[-6:].values[::-1]
    last_lags_seq = x_seq.iloc[-6:].values[::-1]
    pred_month = 8
    pred_year = 2025
    last_mean_3 = np.mean(last_lags_val[:3])
    last_mean_6 = np.mean(last_lags_val[:6])
    
    final_input = np.concatenate([
        last_lags_val, last_lags_seq, [pred_month, pred_year, last_mean_3, last_mean_6]
    ]).reshape(1, -1)
    
    # 예측 수행 (로그 스케일 결과 나옴)
    pred_log_xgb = model_xgb.predict(final_input)[0]
    pred_log_lr = model_lr.predict(final_input)[0]
    
    # [핵심 3] 50:50 섞기
    pred_log_avg = (pred_log_xgb * 0.5) + (pred_log_lr * 0.5)
    
    # [핵심 4] 로그 풀기 (Expm1) -> 원래 금액으로 복구
    pred_val = np.expm1(pred_log_avg)
    
    predictions.append(max(0, int(pred_val)))

elite_df['pred_aug'] = predictions

# ---------------------------------------------------------
# 4. Top-3 평균 전파
# ---------------------------------------------------------
target_values = elite_df.groupby('following_item_id')['pred_aug'].mean()
granger_df['final_value'] = granger_df['following_item_id'].map(target_values).fillna(0)
granger_df['value'] = granger_df['final_value'].round().astype(int)

# 저장
final_submission = granger_df[['leading_item_id', 'following_item_id', 'value']]
filename = 'submission_FINAL_LOG_ENSEMBLE.csv'
final_submission.to_csv(filename, index=False)

print(f"\n✅ [Log 변환 + Linear 앙상블] 파일 생성 완료: {filename}")
print("이 방법은 NMAE(비율 오차)를 줄이는 데 특효약입니다.")

📂 데이터 준비 중...

📊 인과관계 F-score 계산 중...


100%|██████████| 1807/1807 [00:04<00:00, 434.41it/s]



🚀 [Log 변환] XGBoost(50%) + Linear(50%) 앙상블 학습...


100%|██████████| 273/273 [00:03<00:00, 73.73it/s]


✅ [Log 변환 + Linear 앙상블] 파일 생성 완료: submission_FINAL_LOG_ENSEMBLE.csv
이 방법은 NMAE(비율 오차)를 줄이는 데 특효약입니다.


In [34]:
import pandas as pd
import os

# 1. 파일 목록 정의 (방금 만든 파일명들과 일치해야 함)
file_paths = [
    'submission_GRANGER.csv',
    'submission_SPEARMAN.csv',
    'submission_TEST_D_Cointegration.csv', # 혹은 CLEAN 버전
    'submission_TEST_B_MutualInfo.csv'
]

# 데이터프레임 담을 리스트
pair_sets = []

print("📂 파일 로드 및 쌍(Pair) 추출 중...")

for path in file_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        # (선행, 후행) 튜플로 변환하여 집합(Set)으로 저장
        pairs = set(zip(df['leading_item_id'], df['following_item_id']))
        pair_sets.append(pairs)
        print(f"   - {path}: {len(pairs)}개 쌍")
    else:
        print(f"⚠️ 경고: {path} 파일이 없습니다. 이 파일은 건너뜁니다.")

# 2. 교집합 구하기 (Intersection)
if pair_sets:
    # *pair_sets는 리스트의 요소들을 풀어헤쳐서 인자로 넣음
    common_pairs = set.intersection(*pair_sets)
    
    print("-" * 40)
    print(f"🔥 [4대장 교집합] 만장일치로 통과한 쌍: {len(common_pairs)}개")
    print("-" * 40)

    # 3. 결과 데이터프레임 생성
    if len(common_pairs) > 0:
        final_rows = []
        for lead, follow in common_pairs:
            final_rows.append({
                'leading_item_id': lead,
                'following_item_id': follow,
                'value': 0  # 값은 나중에 XGBoost로 채워야 함 (일단 0)
            })
        
        intersection_df = pd.DataFrame(final_rows)
        
        # 저장
        filename = 'submission_INTERSECTION_4_METHODS.csv'
        intersection_df.to_csv(filename, index=False)
        print(f"✅ 파일 생성 완료: {filename}")
        print("👉 이 파일로 다시 XGBoost+Linear 앙상블 코드를 돌려서 값을 채우세요.")
    else:
        print("❌ 교집합이 0개입니다. 4가지 조건이 너무 빡빡한 것 같습니다.")
        print("   전략 수정 추천: 3가지 교집합(Majority Vote)으로 기준을 낮추세요.")
else:
    print("로딩된 파일이 없습니다.")

📂 파일 로드 및 쌍(Pair) 추출 중...
   - submission_GRANGER.csv: 1807개 쌍
   - submission_SPEARMAN.csv: 26개 쌍
   - submission_TEST_D_Cointegration.csv: 6950개 쌍
   - submission_TEST_B_MutualInfo.csv: 477개 쌍
----------------------------------------
🔥 [4대장 교집합] 만장일치로 통과한 쌍: 7개
----------------------------------------
✅ 파일 생성 완료: submission_INTERSECTION_4_METHODS.csv
👉 이 파일로 다시 XGBoost+Linear 앙상블 코드를 돌려서 값을 채우세요.


In [39]:
import pandas as pd
import numpy as np

# 1. 데이터 로드
print("📂 데이터 로드 및 전처리 중...")
train_df = pd.read_csv('train.csv')

# 월별 총합 집계 (Value 기준)
df_grouped = train_df.groupby(['item_id', 'year', 'month'])['value'].sum().reset_index()
df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))

# Pivot Table 생성 (행: 날짜, 열: 아이템, 값: 무역량)
# 결측치는 0으로 채움 (거래 없음)
pivot_df = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)

# 2. 0의 지속성 검증
print("\n🔍 데이터 전수 조사 시작...")

# 한 달 전 데이터 (Shift 1)
prev_month = pivot_df.shift(1)
curr_month = pivot_df

# 데이터가 있는 구간만 비교 (첫 달은 이전 달이 없으니 제외)
# 유효한 마스크: 이전 달 데이터가 NaN이 아닌 경우
valid_mask = prev_month.notnull()

# 조건 1: 지난달이 0이었던 모든 케이스 찾기
# (단, 데이터 수집 시작점이라서 0인 게 아니라, 실제 중간에 0인 경우를 봐야 함)
mask_prev_zero = (prev_month == 0) & valid_mask

# 전체 케이스 수 (지난달 0이었던 횟수)
total_zero_cases = mask_prev_zero.sum().sum()

# 결과 1: 이번 달도 0인 경우 (Zombie Stay)
stay_zero_count = (curr_month[mask_prev_zero] == 0).sum().sum()

# 결과 2: 이번 달에 수입이 발생한 경우 (Resurrected)
resurrect_count = (curr_month[mask_prev_zero] > 0).sum().sum()

# 3. 결과 리포트
ratio_stay = (stay_zero_count / total_zero_cases) * 100
ratio_resurrect = (resurrect_count / total_zero_cases) * 100

print("="*50)
print(f"📊 [팩트 체크] 좀비 이론 검증 결과")
print("="*50)
print(f"총 샘플(지난달 0이었던 경우): {total_zero_cases}건")
print("-" * 50)
print(f"🧟 이번 달도 0 (유지) : {stay_zero_count}건 ({ratio_stay:.2f}%)")
print(f"✨ 이번 달 부활 (발생) : {resurrect_count}건 ({ratio_resurrect:.2f}%)")
print("="*50)

# 4. 반대 케이스 확인 (지난달 거래 있었는데 이번 달 끊길 확률)
mask_prev_active = (prev_month > 0) & valid_mask
total_active_cases = mask_prev_active.sum().sum()
die_count = (curr_month[mask_prev_active] == 0).sum().sum()
ratio_die = (die_count / total_active_cases) * 100

print(f"💡 참고: 거래 하다가 끊길 확률: {ratio_die:.2f}%")

📂 데이터 로드 및 전처리 중...

🔍 데이터 전수 조사 시작...
📊 [팩트 체크] 좀비 이론 검증 결과
총 샘플(지난달 0이었던 경우): 514건
--------------------------------------------------
🧟 이번 달도 0 (유지) : 351건 (68.29%)
✨ 이번 달 부활 (발생) : 163건 (31.71%)
💡 참고: 거래 하다가 끊길 확률: 4.23%


In [42]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# 1. 데이터 로드
print("📂 데이터 로드 중...")
train_df = pd.read_csv('train.csv')
train_df = train_df[train_df['value'] != 0]

# 전처리 (상관계수 계산용)
df_grouped = train_df.groupby(['item_id', 'year', 'month'])['value'].sum().reset_index()
df_grouped['date'] = pd.to_datetime(df_grouped[['year', 'month']].assign(day=1))
pivot_val = df_grouped.pivot(index='date', columns='item_id', values='value').fillna(0)

# 2. 리스트 로드
# (1) 그레인저 (Base)
df_granger = pd.read_csv('submission_GRANGER.csv')
print(f"📊 그레인저 리스트: {len(df_granger)}개 (Base)")

# (2) 공적분 (Source) - 파일명 확인 필요
# 아까 만든 6971개짜리 파일명을 넣으세요. (예: submission_TEST_D_Cointegration.csv)
# 만약 파일이 없으면 로직상 6971개를 다시 계산해야 하지만, 있다고 가정하고 진행
try:
    df_coint = pd.read_csv('submission_TEST_D_Cointegration.csv') 
    # 만약 clean 버전이라면 컬럼이 3개일 테니 그대로 사용
    print(f"📊 공적분 리스트: {len(df_coint)}개 (여기서 알짜를 건져야 함)")
except:
    print("⚠️ 공적분 파일이 없습니다. 코드를 확인해주세요.")
    df_coint = pd.DataFrame()

# 3. 공적분 리스트 정제 (Correlation Filter)
# 전략: 공적분 통과한 애들 중, 피어슨 상관계수가 0.4 이상인 것만 살린다.
# (이유: 상관계수가 너무 낮으면 어차피 XGBoost가 예측을 못해서 오차가 큼 -> 감점)

if not df_coint.empty:
    print("\n⛏️ 공적분 리스트 옥석 가리기 (Correlation > 0.4)...")
    
    valid_coint_pairs = []
    
    for idx, row in tqdm(df_coint.iterrows(), total=len(df_coint)):
        lead = row['leading_item_id']
        follow = row['following_item_id']
        
        y = pivot_val[follow]
        x = pivot_val[lead]
        
        # Lag 1~6 중 최대 상관계수 확인
        max_corr = 0
        for lag in range(1, 7):
            corr = x.shift(lag).corr(y)
            if abs(corr) > abs(max_corr):
                max_corr = abs(corr)
        
        # 기준: 0.4 (이정도는 되어야 예측 가능)
        if max_corr > 0.4:
            row['max_corr'] = max_corr
            valid_coint_pairs.append(row)
    
    df_coint_filtered = pd.DataFrame(valid_coint_pairs)
    print(f"   -> 6971개 중 {len(df_coint_filtered)}개 생존 (Correlation > 0.4)")

    # 4. 합치기 (Union)
    # 그레인저(전체) + 공적분(필터링됨)
    print("\n🔗 리스트 합치기 (Union)...")
    
    # 컬럼 통일
    cols = ['leading_item_id', 'following_item_id']
    
    combined = pd.concat([
        df_granger[cols],
        df_coint_filtered[cols]
    ])
    
    # 중복 제거 (그레인저랑 공적분 둘 다 찾은 거 중복 방지)
    final_list = combined.drop_duplicates()
    
    # 값(Value) 컬럼 추가 (일단 0으로, 나중에 XGBoost로 채워야 함)
    final_list['value'] = 99999999999
    
    print("-" * 50)
    print(f"✅ 최종 후보 개수: {len(final_list)}개")
    print(f"   (그레인저 {len(df_granger)} + 공적분 구조대 {len(final_list) - len(df_granger)})")
    print("-" * 50)
    
    # 저장
    filename = 'submission_FINAL_UNION_STRATEGY.csv'
    final_list.to_csv(filename, index=False)
    print(f"파일 저장 완료: {filename}")
    
else:
    print("공적분 데이터가 없어서 합치기 실패.")

📂 데이터 로드 중...
📊 그레인저 리스트: 1807개 (Base)
📊 공적분 리스트: 6950개 (여기서 알짜를 건져야 함)

⛏️ 공적분 리스트 옥석 가리기 (Correlation > 0.4)...


100%|██████████| 6950/6950 [00:02<00:00, 3204.17it/s]

   -> 6971개 중 1142개 생존 (Correlation > 0.4)

🔗 리스트 합치기 (Union)...
--------------------------------------------------
✅ 최종 후보 개수: 2298개
   (그레인저 1807 + 공적분 구조대 491)
--------------------------------------------------
파일 저장 완료: submission_FINAL_UNION_STRATEGY.csv
